# Entraînement Deep Learning — prédiction des compétences IA

Ce notebook entraîne un modèle **CamemBERT** pour prédire plusieurs compétences IA à partir des informations d'une formation.

La tâche est une **classification multi-étiquette** : une formation peut correspondre à plusieurs compétences simultanément.

Points importants :

- le jeu contient peu de données, donc on affine un modèle préentraîné au lieu d'entraîner un réseau depuis zéro ;
- les doublons sont séparés par intitulé de formation pour éviter les fuites de données ;
- les métriques principales sont le F1 micro et le F1 macro ;
- le seuil de décision est ajusté sur le jeu de validation.


In [ ]:
# À exécuter une seule fois dans l'environnement Python du notebook
%pip install -U pandas numpy scikit-learn torch transformers datasets accelerate sentencepiece matplotlib


In [ ]:
from pathlib import Path
import re
import json
import unicodedata
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

print("PyTorch :", torch.__version__)
print("CUDA disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))


## 1. Chargement du jeu de données

Place le fichier CSV dans le même dossier que ce notebook, ou modifie `CSV_PATH`.


In [ ]:
CSV_PATH = Path("Dataset_V7_Anton_CSV - Dataset_V7_Anton_CSV.csv.csv")

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Fichier introuvable : {CSV_PATH.resolve()}\n"
        "Modifie CSV_PATH avec le chemin réel du fichier."
    )

df = pd.read_csv(CSV_PATH)

print("Dimensions :", df.shape)
display(df.head(3))
print("\nColonnes :")
for col in df.columns:
    print("-", col)


## 2. Nettoyage et construction du texte d'entrée

La cible est la colonne `Compétences IA extraites`.

Le modèle utilisera :

- l'intitulé ;
- le type de certification ;
- le niveau ;
- les codes ROME ;
- la modalité ;
- la durée ;
- le public cible.

Les tags TrendRadar ne sont volontairement pas utilisés comme entrée, car ils contiennent souvent directement la réponse attendue.


In [ ]:
TARGET_COLUMN = "Compétences IA extraites"

INPUT_COLUMNS = [
    "Intitulé de la formation",
    "Type de certification",
    "Niveau",
    "Codes ROME",
    "Modalité",
    "Durée",
    "Public cible",
]

def normalize_text(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    value = re.sub(r"\s+", " ", value)
    return value

def normalize_group_title(value):
    value = normalize_text(value).lower()
    value = unicodedata.normalize("NFKD", value)
    value = "".join(c for c in value if not unicodedata.combining(c))
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()

def parse_labels(value):
    if pd.isna(value):
        return []
    labels = []
    for item in str(value).split("|"):
        item = item.strip()
        if item:
            labels.append(item)
    return sorted(set(labels))

def build_input_text(row):
    parts = []
    for column in INPUT_COLUMNS:
        value = normalize_text(row.get(column, ""))
        if value and value != "0":
            parts.append(f"{column} : {value}")
    return " [SEP] ".join(parts)

data = df.copy()
data["labels_text"] = data[TARGET_COLUMN].apply(parse_labels)
data["text"] = data.apply(build_input_text, axis=1)
data["group_title"] = data["Intitulé de la formation"].apply(normalize_group_title)

data = data[data["labels_text"].map(len) > 0].reset_index(drop=True)

print("Nombre de lignes utilisables :", len(data))
print("Nombre d'intitulés uniques :", data["group_title"].nunique())
print("Exemple de texte :\n")
print(data.loc[0, "text"])
print("\nCompétences :", data.loc[0, "labels_text"])


## 3. Analyse des étiquettes

In [ ]:
from collections import Counter

RARE_SKILL_THRESHOLD = 20
REQUIRED_COLUMNS = {'statut_annotation', 'competences_ia'}


def _normalize_competence_for_counting(value):
    """Normalise une compétence sans modifier `train_df`."""
    if value is None:
        return ''
    text = str(value).strip()
    if not text:
        return ''
    text = ' '.join(text.replace('\u00a0', ' ').split())
    text = text.strip(" |;,-•\t\n\r")
    if not text:
        return ''
    upper = text.upper()
    if upper in {'NLP', 'RAG', 'LLM', 'MLOPS', 'API', 'SQL', 'HTTP', 'REST'}:
        return upper
    # Uniformisation raisonnable sans casser les acronymes
    return text[0].upper() + text[1:]


def _ensure_parse_multi_values():
    if 'parse_multi_values' not in globals() or not callable(parse_multi_values):
        raise NameError(
            "La fonction `parse_multi_values` doit être définie avant cette cellule."
        )
    probe = parse_multi_values('Python | FastAPI')
    if not isinstance(probe, list):
        raise TypeError(
            "`parse_multi_values` doit retourner une liste. Valeur reçue: " + str(type(probe))
        )


missing_columns = REQUIRED_COLUMNS.difference(train_df.columns)
if missing_columns:
    raise KeyError(
        "Colonnes manquantes dans `train_df` : " + ", ".join(sorted(missing_columns))
    )

_ensure_parse_multi_values()

binary_counts = train_df['statut_annotation'].value_counts(dropna=False)
print('Distribution de statut_annotation :')
display(binary_counts.to_frame(name='occurrences'))

ia_train = train_df[train_df['statut_annotation'] == 'ia_confirmee'].copy()
if ia_train.empty:
    print("Aucune formation `ia_confirmee` n'est présente dans `train_df`.")
    comp_stats = pd.DataFrame(
        columns=['competence', 'occurrences', 'pourcentage_formations_ia', 'est_rare']
    )
    display(comp_stats)
else:
    total_ia_formations = len(ia_train)
    competence_counter = Counter()

    for raw_value in ia_train['competences_ia']:
        parsed = parse_multi_values(raw_value if pd.notna(raw_value) else '')
        if not isinstance(parsed, list):
            raise TypeError("`parse_multi_values` doit retourner une liste de compétences.")
        normalized_skills = []
        for item in parsed:
            normalized = _normalize_competence_for_counting(item)
            if normalized:
                normalized_skills.append(normalized)
        for competence in set(normalized_skills):
            competence_counter[competence] += 1

    comp_stats = pd.DataFrame(
        sorted(competence_counter.items(), key=lambda x: (-x[1], x[0])),
        columns=['competence', 'occurrences'],
    )
    comp_stats['pourcentage_formations_ia'] = (
        comp_stats['occurrences'] / total_ia_formations * 100
    ).round(2)
    comp_stats['est_rare'] = comp_stats['occurrences'] < RARE_SKILL_THRESHOLD

    print(f"Nombre total de compétences distinctes : {len(comp_stats)}")
    print(f"Nombre de formations IA confirmées : {total_ia_formations}")
    print()
    print('20 compétences les plus fréquentes :')
    display(comp_stats.head(20))

    rare_skills = comp_stats[comp_stats['est_rare']].copy()
    print()
    print(f'Compétences rares (< {RARE_SKILL_THRESHOLD} occurrences) :')
    display(rare_skills)


## 4. Découpage sans fuite de données

Les lignes ayant le même intitulé normalisé restent dans le même sous-ensemble.

Répartition visée :

- 70 % entraînement ;
- 15 % validation ;
- 15 % test.


In [ ]:
RANDOM_STATE = 42

groups = data["group_title"]

first_split = GroupShuffleSplit(
    n_splits=1,
    train_size=0.70,
    random_state=RANDOM_STATE,
)

train_idx, temp_idx = next(
    first_split.split(data, groups=groups)
)

train_df = data.iloc[train_idx].reset_index(drop=True)
temp_df = data.iloc[temp_idx].reset_index(drop=True)

second_split = GroupShuffleSplit(
    n_splits=1,
    train_size=0.50,
    random_state=RANDOM_STATE,
)

val_idx, test_idx = next(
    second_split.split(temp_df, groups=temp_df["group_title"])
)

val_df = temp_df.iloc[val_idx].reset_index(drop=True)
test_df = temp_df.iloc[test_idx].reset_index(drop=True)

print("Train :", len(train_df), "lignes,", train_df["group_title"].nunique(), "groupes")
print("Validation :", len(val_df), "lignes,", val_df["group_title"].nunique(), "groupes")
print("Test :", len(test_df), "lignes,", test_df["group_title"].nunique(), "groupes")

assert set(train_df["group_title"]).isdisjoint(set(val_df["group_title"]))
assert set(train_df["group_title"]).isdisjoint(set(test_df["group_title"]))
assert set(val_df["group_title"]).isdisjoint(set(test_df["group_title"]))

print("Aucune fuite de groupe détectée.")


## 5. Encodage multi-étiquette

In [ ]:
mlb = MultiLabelBinarizer()
mlb.fit(data["labels_text"])

LABEL_NAMES = list(mlb.classes_)
NUM_LABELS = len(LABEL_NAMES)

def add_binary_labels(frame):
    frame = frame.copy()
    encoded = mlb.transform(frame["labels_text"])
    frame["labels"] = [row.astype(np.float32).tolist() for row in encoded]
    return frame

train_df = add_binary_labels(train_df)
val_df = add_binary_labels(val_df)
test_df = add_binary_labels(test_df)

id2label = {i: label for i, label in enumerate(LABEL_NAMES)}
label2id = {label: i for i, label in id2label.items()}

print("Nombre de classes :", NUM_LABELS)
print(LABEL_NAMES)


## 6. Tokenisation avec CamemBERT

In [ ]:
MODEL_NAME = "camembert-base"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(frame):
    return Dataset.from_pandas(
        frame[["text", "labels"]],
        preserve_index=False,
    )

train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)
test_ds = to_hf_dataset(test_df)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

columns = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=columns)
val_ds.set_format(type="torch", columns=columns)
test_ds.set_format(type="torch", columns=columns)

train_ds


## 7. Création et entraînement du modèle

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
    problem_type="multi_label_classification",
)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

DECISION_THRESHOLD = 0.50

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probabilities = sigmoid(logits)
    predictions = (probabilities >= DECISION_THRESHOLD).astype(int)

    return {
        "f1_micro": f1_score(labels, predictions, average="micro", zero_division=0),
        "f1_macro": f1_score(labels, predictions, average="macro", zero_division=0),
        "precision_micro": precision_score(labels, predictions, average="micro", zero_division=0),
        "recall_micro": recall_score(labels, predictions, average="micro", zero_division=0),
    }

training_args = TrainingArguments(
    output_dir="./camembert_competences_ia_checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=12,
    weight_decay=0.01,
    warmup_ratio=0.10,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=RANDOM_STATE,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

trainer.train()


## 8. Recherche du meilleur seuil

Le seuil 0,50 n'est pas toujours optimal en classification multi-étiquette. On choisit le meilleur seuil uniquement avec la validation.


In [ ]:
val_output = trainer.predict(val_ds)
val_probabilities = sigmoid(val_output.predictions)
val_labels = val_output.label_ids.astype(int)

threshold_results = []

for threshold in np.arange(0.10, 0.91, 0.05):
    val_predictions = (val_probabilities >= threshold).astype(int)
    threshold_results.append({
        "threshold": round(float(threshold), 2),
        "f1_micro": f1_score(
            val_labels,
            val_predictions,
            average="micro",
            zero_division=0,
        ),
        "f1_macro": f1_score(
            val_labels,
            val_predictions,
            average="macro",
            zero_division=0,
        ),
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df)

best_row = threshold_df.loc[threshold_df["f1_micro"].idxmax()]
BEST_THRESHOLD = float(best_row["threshold"])

print("Meilleur seuil :", BEST_THRESHOLD)
print("F1 micro validation :", best_row["f1_micro"])


## 9. Évaluation finale sur le jeu de test

In [ ]:
test_output = trainer.predict(test_ds)
test_probabilities = sigmoid(test_output.predictions)
test_labels = test_output.label_ids.astype(int)
test_predictions = (test_probabilities >= BEST_THRESHOLD).astype(int)

print("F1 micro :", f1_score(
    test_labels,
    test_predictions,
    average="micro",
    zero_division=0,
))
print("F1 macro :", f1_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0,
))
print("Précision micro :", precision_score(
    test_labels,
    test_predictions,
    average="micro",
    zero_division=0,
))
print("Rappel micro :", recall_score(
    test_labels,
    test_predictions,
    average="micro",
    zero_division=0,
))

report = classification_report(
    test_labels,
    test_predictions,
    target_names=LABEL_NAMES,
    zero_division=0,
    output_dict=True,
)

report_df = pd.DataFrame(report).T
display(report_df)


## 10. Inspection des prédictions

In [ ]:
def labels_from_binary(binary_vector):
    return [
        LABEL_NAMES[i]
        for i, value in enumerate(binary_vector)
        if value == 1
    ]

results = test_df[["text", "labels_text"]].copy()
results["predictions"] = [
    labels_from_binary(row)
    for row in test_predictions
]
results["probabilites_max"] = test_probabilities.max(axis=1)

display(results.head(20))


## 11. Sauvegarde du modèle

In [ ]:
MODEL_DIR = Path("./modele_camembert_competences_ia")
MODEL_DIR.mkdir(exist_ok=True)

trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

metadata = {
    "labels": LABEL_NAMES,
    "threshold": BEST_THRESHOLD,
    "max_length": MAX_LENGTH,
    "base_model": MODEL_NAME,
    "input_columns": INPUT_COLUMNS,
}

with open(MODEL_DIR / "metadata.json", "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

print("Modèle sauvegardé dans :", MODEL_DIR.resolve())


## 12. Fonction de prédiction sur une nouvelle formation

In [ ]:
def predict_skills(text, threshold=None):
    if threshold is None:
        threshold = BEST_THRESHOLD

    model.eval()

    encoded = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
    )

    device = model.device
    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    with torch.no_grad():
        logits = model(**encoded).logits

    probabilities = torch.sigmoid(logits)[0].cpu().numpy()

    predictions = [
        {
            "competence": LABEL_NAMES[i],
            "probabilite": round(float(probability), 4),
        }
        for i, probability in enumerate(probabilities)
        if probability >= threshold
    ]

    return sorted(
        predictions,
        key=lambda item: item["probabilite"],
        reverse=True,
    )

example_text = """
Intitulé de la formation : Développeur en intelligence artificielle
Niveau : BAC+5
Modalité : distance
Public cible : développeurs souhaitant apprendre Python, le machine learning,
le deep learning, le traitement du langage et le déploiement de modèles.
"""

predict_skills(example_text)


## Recommandations

Ce jeu de données est encore petit et très déséquilibré. Les compétences rares, par exemple `Computer Vision`, `Reinforcement Learning` ou `Gestion de projet IA`, risquent d'être mal apprises.

Pour améliorer réellement le modèle :

1. augmenter le nombre d'intitulés uniques ;
2. corriger les doublons et variantes orthographiques ;
3. vérifier manuellement les compétences cibles ;
4. obtenir au moins 30 à 50 exemples distincts par compétence ;
5. comparer ce modèle à une baseline TF-IDF + régression logistique multi-étiquette.
